# The Same Pipeline, on Text We Can Share

**Computational Text Analysis Pack · Module 13 — Case Studies**

*Technique:* Reproducing the case study's six steps end to end in Keras

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/16-case-studies/13d_the_pipeline_on_clean_text.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The case study's six steps, run end to end on MasakhaNEWS — a corpus this course
can hand you, in sixteen African languages, openly licensed.

The task changes from "which novelist wrote this passage" to "which topic is
this article", but every step is the one the original pipeline used, including
the group-aware split discipline from 13b.

## The tooling

**Keras** on TensorFlow, which is installed here. The whole model is about
32,000 parameters and trains on CPU in seconds — no GPU, no pod, no API key.
Module 23 is where the large pretrained models come in; this is the small,
transparent end of the same family.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
# Step 1-2: load and split. Stratified, and we check the classes survive.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

sub = df[df["n_words"] >= 30].reset_index(drop=True)
train_df, test_df = train_test_split(sub, test_size=0.25, random_state=0,
                                     stratify=sub["category"])
print(f"{len(train_df):,} train / {len(test_df):,} test")
print(train_df["category"].value_counts().to_string())

In [ ]:
# Step 3: vectorise. TextVectorization adapts on TRAINING text only — fitting
# it on everything is the vocabulary-leak module 7 warned about.
from tensorflow.keras.layers import TextVectorization

MAX_WORDS, MAX_LEN, DIM, EPOCHS = 10000, 200, 32, 15

vec = TextVectorization(max_tokens=MAX_WORDS, output_sequence_length=MAX_LEN)
vec.adapt(train_df["text"].values)
x_train = vec(train_df["text"].values).numpy()
x_test = vec(test_df["text"].values).numpy()

lb = LabelBinarizer()
y_train = lb.fit_transform(train_df["category"])
y_test = lb.transform(test_df["category"])
print("x_train", x_train.shape, "| x_test", x_test.shape,
      "| classes", [str(c) for c in lb.classes_])
print("\nfirst 15 vocabulary entries:", [str(w) for w in vec.get_vocabulary()[:15]])

In [ ]:
# Step 4: build. Input first, so summary() reports a built model.
from tensorflow.keras import Input, Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D

model = Sequential([
    Input(shape=(MAX_LEN,), dtype="int32"),
    Embedding(input_dim=MAX_WORDS, output_dim=DIM),
    GlobalAveragePooling1D(),
    Dense(len(lb.classes_), activation="softmax"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
# Step 5: train.
import time
t0 = time.time()
history = model.fit(x_train, y_train, validation_split=0.2,
                    epochs=EPOCHS, verbose=0)
print(f"trained in {time.time() - t0:.0f}s on CPU")
print(f"final train acc {history.history['accuracy'][-1]:.3f} | "
      f"val acc {history.history['val_accuracy'][-1]:.3f}")

plt.figure(figsize=(6.5, 3.8))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend()
plt.title("Watch for the gap opening — that is overfitting")
plt.tight_layout()

In [ ]:
# Step 6: evaluate, against the baselines this course already established.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

loss, acc = model.evaluate(x_test, y_test, verbose=0)
majority = test_df["category"].value_counts(normalize=True).max()

tv = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=3, max_features=20000,
                     sublinear_tf=True)
A = tv.fit_transform(train_df["text"]); B = tv.transform(test_df["text"])
lr = LogisticRegression(max_iter=1000).fit(A, train_df["category"])
lr_acc = accuracy_score(test_df["category"], lr.predict(B))

print(f"{'majority-class baseline':<34}{majority:.3f}")
print(f"{'Keras embedding-average':<34}{acc:.3f}")
print(f"{'TF-IDF + logistic regression':<34}{lr_acc:.3f}")
print()
if lr_acc > acc:
    print("The classical model wins. That is the usual result at this scale, and")
    print("it is the honest headline of this whole module: the neural machinery")
    print("is not free, and on a few thousand documents it does not pay for")
    print("itself. Module 23 shows what changes when the model arrives")
    print("pretrained on billions of words instead.")

In [ ]:
names = [str(c) for c in lb.classes_]
cm = confusion_matrix(np.argmax(y_test, axis=1), np.argmax(model.predict(x_test, verbose=0), axis=1))
print("rows = true category, columns = predicted\n")
print(pd.DataFrame(cm, index=names, columns=names).to_string())

### 🔧 Your turn 1

Set MAX_WORDS to 1000 and re-run. Does a smaller vocabulary help or hurt here, and why is that the opposite of what it did for authorship in the case study?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

It usually *hurts* topic classification, which is the reverse of the novels
result. For authorship, rare words are character and place names that identify
the book rather than the author, so capping the vocabulary removed noise. For
topic, rare words are exactly the content words that distinguish health from
sport, so capping removes the signal. Same knob, opposite direction, because the
task changed — which is why "use a vocabulary of N" is never transferable advice
and every hyperparameter needs justifying against the specific question.

</details>